## Data Quality Verification

The dataset was manually verified prior to automated checks. To further ensure the reliability of the parallel Finnish-English course descriptions, cosine similarity scores were computed between each Finnish field and its DeepL-translated English equivalent using `paraphrase-multilingual-mpnet-base-v2`. Pairs scoring below 0.75 were flagged for a second round of manual review. The verification covered four fields: `title_fi`, `outcomes_fi`, `contents_fi`, and `assessment_fi`.

In [ ]:
import deepl
import pandas as pd
from sentence_transformers import SentenceTransformer, util

df = pd.read_csv(r"C:\Users\madee\OneDrive\Desktop\thesis\data\raw\final_dataset.csv")
translator = deepl.Translator("eb677f3a-544a-4b74-8b02-5ae69da32ca7:fx")
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

### Translation and Similarity Scoring

Each Finnish field was translated to English using the DeepL API. Cosine similarity was then computed between the DeepL translation and the original English field using sentence embeddings. This produces an alignment score per course per field, with lower scores indicating potential misalignment.

In [ ]:
fi_en_pairs = [
    ("title_fi", "title_en"),
    ("outcomes_fi", "outcomes_en"),
    ("contents_fi", "contents_en"),
    ("assessment_fi", "assessment_en"),
]

def alignment_score(text1, text2):
    emb1 = model.encode(str(text1))
    emb2 = model.encode(str(text2))
    return util.cos_sim(emb1, emb2).item()

def translate(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""
    return translator.translate_text(str(text), target_lang="EN-US").text

for fi_col, en_col in fi_en_pairs:
    translated_col = f"{fi_col}_translated"
    score_col = f"{fi_col}_score"
    df[translated_col] = df[fi_col].apply(translate)
    df[score_col] = df.apply(
        lambda row: alignment_score(row[en_col], row[translated_col]), axis=1
    )

### Flagged Pairs at Threshold 0.75

Pairs with a cosine similarity score below 0.75 were flagged for manual review. The threshold of 0.75 was selected based on standard practice in semantic similarity evaluation, where lower values indicate potential misalignment rather than natural translation variation.

In [ ]:
threshold = 0.75

for fi_col, _ in fi_en_pairs:
    score_col = f"{fi_col}_score"
    low = df[df[score_col] < threshold][["course_id", score_col]]
    print(f"\nLow confidence in {fi_col} ({len(low)} pairs):")
    print(low)

### Inspection of Flagged Title Pairs

Nine pairs were flagged in the `title_fi` field. All were retrieved and manually reviewed below.

In [ ]:
flagged_ids = [
    "TT00CD56", "TT00CD58", "TT00CJ28", "TT00CD75",
    "TT00CD88", "TT00CE16", "TT00CE23", "TT00CE29", "TT00CE28"
]

flagged_titles = df[df["course_id"].isin(flagged_ids)][["course_id", "title_fi", "title_en", "title_fi_score"]]
print(flagged_titles.to_string(index=False))

All 9 flagged title pairs were manually reviewed and confirmed as correctly aligned. The low scores fell into two recurring patterns:

- **Abbreviated English titles**: Some English titles omit words present in the Finnish original (e.g., *"Mat1 Yhtälöt tukiopinnot"* vs *"Math1 Support"*), reducing lexical overlap without reflecting misalignment.
- **Finnish compound words vs multi-word English equivalents**: Finnish regularly combines concepts into a single compound word (e.g., *"Konesaliverkot"*, *"Takaisinmallintaminen"*), while English expresses the same concept across multiple words (e.g., *"Data Center Networks"*, *"Reverse Engineering"*). This structural difference reduces embedding similarity even when meaning is identical.

The most extreme case was TT00CE29 (*"Konesaliverkot"* vs *"Data Center Networks"*, score: -0.035). Manual inspection of the full course record confirmed the pair is correctly aligned. The near-zero score is attributable entirely to the single-word vs multi-word mismatch. Title fields were excluded from the main similarity evaluation accordingly.

### Inspection of Flagged Assessment Pairs

Three pairs were flagged in the `assessment_fi` field. Their content was retrieved and reviewed against DeepL translations.

In [ ]:
flagged_assessment_ids = ["TT00CD94", "TT00CE05", "TT00CE06"]

flagged_assessment = df[df["course_id"].isin(flagged_assessment_ids)][["course_id", "assessment_fi", "assessment_en", "assessment_fi_score"]]
print(flagged_assessment.to_string(index=False))

### Data Quality Verification Summary

Three course pairs (TT00CD94, TT00CE05, TT00CE06) were flagged below the 0.75 cosine similarity threshold in the assessment field. Manual review confirmed all three as correctly aligned parallel pairs. The reduced scores were attributed to inconsistent translation of Finnish grade labels. For example, the Finnish grading scale words were translated as *"Passing"* or *"Fair"* by DeepL, while the original English descriptions used *"Sufficient"*. Similarly, *"Kiitettävä"* appeared as both *"Excellent"* and *"Very good"* across sources. These label-level discrepancies introduced enough surface variation to lower the embedding score without reflecting any actual content misalignment. The substantive assessment criteria at each grade level were identical across all three pairs.

Overall, the `outcomes_fi` and `contents_fi` fields produced zero flagged pairs at the 0.75 threshold, confirming that the core fields used in the thesis experiments are clean and well-aligned.

In [ ]:
df.to_csv("dataset_verified.csv", index=False)
print("Verified dataset saved to dataset_verified.csv")